In [1]:
import json

def load_marker_dataset(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed = []

    for i, item in enumerate(data):
        text = item.get("text", None)
        if text is None:
            continue

        spans = item.get("spans", [])
        marker_spans = []

        for span in spans:
            labels = span.get("labels", [])
            if "MARKER" in labels:
                marker_spans.append(
                    (span["start"], span["end"])
                )

        if len(marker_spans) == 0:
            continue

        processed.append({
            "text": text,
            "spans": marker_spans
        })

    return processed


dataset_raw = load_marker_dataset("meged3.json")
print(f"Loaded {len(dataset_raw)} examples with markers")

Loaded 3223 examples with markers


In [2]:
from transformers import AutoTokenizer

model_ckpt = "Eraly-ml/KazBERT"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

label_list = ["O", "B-MARKER", "I-MARKER"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

c:\Users\egisb\voice1\kazFineTune\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
def tokenize_and_align(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_offsets_mapping=True
    )

    labels = []
    offsets = tokenized["offset_mapping"]

    for offset in offsets:
        if offset == (0, 0):
            labels.append(-100)
            continue

        token_start, token_end = offset
        assigned = "O"

        for span_start, span_end in example["spans"]:
            overlap = token_end > span_start and token_start < span_end
            if overlap:
                assigned = "B-MARKER" if assigned == "O" else "I-MARKER"
                break

        labels.append(label2id[assigned])

    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")

    return tokenized

In [5]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(dataset_raw, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

train_dataset = train_dataset.map(tokenize_and_align)
test_dataset = test_dataset.map(tokenize_and_align)

Map: 100%|██████████| 645/645 [00:00<00:00, 3374.12 examples/s]


In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at Eraly-ml/KazBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                true_labels.append(l_)
                true_preds.append(p_)

    f1 = f1_score(true_labels, true_preds, average="macro")
    return {"macro_f1": f1}

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./kazbert_marker_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    fp16=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\egisb\AppData\Local\Temp\ipykernel_15020\4223154140.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.008702,0.994925
2,0.042700,0.005863,0.997181
3,0.042700,0.007526,0.995873
4,0.003200,0.007917,0.996064
5,0.001300,0.007605,0.996435


TrainOutput(global_step=1615, training_loss=0.014722568414897741, metrics={'train_runtime': 202.1243, 'train_samples_per_second': 63.773, 'train_steps_per_second': 7.99, 'total_flos': 842036411312640.0, 'train_loss': 0.014722568414897741, 'epoch': 5.0})

In [10]:
from transformers import AutoTokenizer

text = "Machine learning is awesome!! Thanks KGP Talkie."

model_ckpt = "Eraly-ml/KazBERT"
distilbert_tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [11]:
trainer.save_model("kazbert_marker_model_final")
distilbert_tokenizer.save_pretrained("kazbert_marker_model_final")

('kazbert_marker_model_final\\tokenizer_config.json',
 'kazbert_marker_model_final\\special_tokens_map.json',
 'kazbert_marker_model_final\\vocab.txt',
 'kazbert_marker_model_final\\added_tokens.json',
 'kazbert_marker_model_final\\tokenizer.json')

In [12]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_dir = "kazbert_marker_model_final"  
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForTokenClassification.from_pretrained(model_dir)

In [13]:
marker_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first"  
)

Device set to use cuda:0


In [14]:
sentence = "Күн бұзылғандықтан біз үйге қайттық."
predictions = marker_pipeline(sentence)
print(predictions)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'MARKER', 'score': np.float32(0.9993961), 'word': 'бұзылғандықтан', 'start': 4, 'end': 18}]
